# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata using the API object (not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Explore available record sets, each field's `@id`, and columns in the dataset schema.

Each entity is referenced by its `@id`, ensuring clarity and accuracy when handling the schema.

In [ ]:
# List record sets
record_sets = dataset.schema.record_sets

print("Record Sets found:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}, Description: {getattr(rs, 'description', 'N/A')}")

# For each record set, list fields and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field: {field.name}, @id: {field.id}, DataType: {getattr(field, 'data_type', 'N/A')}")
        if hasattr(field, 'column') and field.column:
            print(f"    - Column @id: {field.column.id}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.

We use the record set and field `@id`s provided above to reference specifically the main survey data.

In [ ]:
# Select the primary record set (usually named 'MentalHealthSurvey', 'SurveyResponses' or similar)
main_record_set_id = None
for rs in dataset.schema.record_sets:
    # Heuristic: look for a record set that contains survey or mental health responses
    if 'mental' in rs.name.lower() or 'survey' in rs.name.lower():
        main_record_set_id = rs.id
        break

if main_record_set_id is None:
    # Fallback to first record set
    main_record_set_id = dataset.schema.record_sets[0].id
    print(f"No explicit survey record set found. Using: {main_record_set_id}")

# Optionally, list all available record set IDs
record_set_ids = [rs.id for rs in dataset.schema.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for the main record set
print("Columns in the main record set:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes.

All references to columns and fields use their `@id` for clarity and reproducibility.

In [ ]:
# Example: Analyze GAD-7 scores (Generalized Anxiety Disorder assessment)
# Find a numeric field (e.g., GAD-7 score column) using the field @id
survey_df = dataframes[main_record_set_id]

# Heuristic: Try to find GAD-7 column
gad7_field_id = None
for field in dataset.schema.record_sets[0].fields:
    if 'gad' in field.name.lower():
        gad7_field_id = field.id
        break

# If not found, pick the first numeric column
if gad7_field_id is None:
    for col in survey_df.columns:
        if 'gad' in col.lower():
            gad7_field_id = col
            break

if gad7_field_id is None:
    print("GAD-7 field not found. Please check field names.")

# Set threshold
threshold = 10
if gad7_field_id is not None and gad7_field_id in survey_df.columns:
    filtered_df = survey_df[survey_df[gad7_field_id] > threshold].copy()
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize score
    normalized_col = f"{gad7_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, normalized_col]].head())

    # Group by demographic attribute, e.g., gender
    # Try to find gender field @id
    gender_field_id = None
    for field in dataset.schema.record_sets[0].fields:
        if 'gender' in field.name.lower():
            gender_field_id = field.id
            break
    if gender_field_id is None:
        # fallback
        for col in survey_df.columns:
            if 'gender' in col.lower():
                gender_field_id = col
                break

    if gender_field_id and gender_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_field_id)[gad7_field_id].mean().reset_index()
        print(f"Grouped data by {gender_field_id} (mean {gad7_field_id}):")
        print(grouped_df.head())
else:
    print("GAD-7 score field not available for EDA.")

## 5. Visualization
Visualize distributions and relationships, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id is not None and gad7_field_id in survey_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(survey_df[gad7_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {gad7_field_id} (GAD-7 Scores)")
    plt.xlabel(gad7_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if gender_field_id and gender_field_id in survey_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=survey_df[gender_field_id], y=survey_df[gad7_field_id])
        plt.title(f"{gad7_field_id} Distribution by {gender_field_id}")
        plt.xlabel(gender_field_id)
        plt.ylabel(gad7_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` library and the Croissant schema to programmatically explore and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya.

Key findings:
- The dataset contains valuable survey and demographic information referenced precisely with `@id`s.
- EDA steps included filtering, normalization, and grouping, showcasing how to handle numeric and categorical data.
- Visualizations revealed distributions and potential disparities across demographic groups.

These tools and practices support reproducible, FAIR AI-ready data analytics.